In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

year = "2024"

In [2]:
standings_url =  "https://fbref.com/en/comps/9/Premier-League-Stats"

In [3]:
years = list(range(2024, 2022, -1))
all_matches = []

In [4]:
standings_url =  "https://fbref.com/en/comps/9/Premier-League-Stats"

In [5]:

for year in years:
    data = requests.get(standings_url)
    soup = BeautifulSoup(data.text)
    standings_table = soup.select('table.stats_table')[0]

    links = [l.get("href") for l in standings_table.find_all('a')]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]
    
    previous_season = soup.select("a.prev")[0].get("href")
    standings_url = f"https://fbref.com{previous_season}"
    
    time.sleep(1)
    for team_url in team_urls:
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        data = requests.get(team_url)
        matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
        soup = BeautifulSoup(data.text)
        links = [l.get("href") for l in soup.find_all('a')]
        links = [l for l in links if l and 'all_comps/shooting/' in l]
        data = requests.get(f"https://fbref.com{links[0]}")
        shooting = pd.read_html(data.text, match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
        team_data = team_data[team_data["Comp"] == "Premier League"]
        
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)
        time.sleep(12)#tiene este tiempo para cumplir con las politicas de la pagina web

C:\Users\edgar\AppData\Local\Temp\ipykernel_9684\1278523688.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
C:\Users\edgar\AppData\Local\Temp\ipykernel_9684\1278523688.py:22: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  shooting = pd.read_html(data.text, match="Shooting")[0]
C:\Users\edgar\AppData\Local\Temp\ipykernel_9684\1278523688.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
C:\Users\edgar\AppData\Local\Temp\ipykernel_9684\1278523688.py:22: FutureWarning: Passing literal html

In [6]:
len(all_matches)

40

In [7]:
match_df = pd.concat(all_matches)

In [8]:
match_df.columns = [c.lower() for c in match_df.columns]

In [9]:
match_df

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,match report,notes,sh,sot,dist,fk,pk,pkatt,season,team
0,2024-08-17,12:30,Premier League,Matchweek 1,Sat,Away,W,2.0,0.0,Ipswich Town,...,Match Report,NaN,18.0,5.0,14.8,0.0,0,0,2024,Liverpool
1,2024-08-25,16:30,Premier League,Matchweek 2,Sun,Home,W,2.0,0.0,Brentford,...,Match Report,NaN,19.0,8.0,13.6,1.0,0,0,2024,Liverpool
2,2024-09-01,16:00,Premier League,Matchweek 3,Sun,Away,W,3.0,0.0,Manchester Utd,...,Match Report,NaN,11.0,3.0,13.4,0.0,0,0,2024,Liverpool
3,2024-09-14,15:00,Premier League,Matchweek 4,Sat,Home,L,0.0,1.0,Nott'ham Forest,...,Match Report,NaN,14.0,5.0,14.9,0.0,0,0,2024,Liverpool
5,2024-09-21,15:00,Premier League,Matchweek 5,Sat,Home,W,3.0,0.0,Bournemouth,...,Match Report,NaN,19.0,12.0,16.6,0.0,0,0,2024,Liverpool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36,2024-04-24,20:00,Premier League,Matchweek 29,Wed,Away,L,2,4,Manchester Utd,...,Match Report,NaN,10.0,4.0,17.8,1.0,0,0,2023,Sheffield United
37,2024-04-27,15:00,Premier League,Matchweek 35,Sat,Away,L,1,5,Newcastle Utd,...,Match Report,NaN,15.0,4.0,13.5,0.0,0,0,2023,Sheffield United
38,2024-05-04,15:00,Premier League,Matchweek 36,Sat,Home,L,1,3,Nott'ham Forest,...,Match Report,NaN,16.0,4.0,18.0,0.0,1,1,2023,Sheffield United
39,2024-05-11,15:00,Premier League,Matchweek 37,Sat,Away,L,0,1,Everton,...,Match Report,NaN,13.0,1.0,21.0,0.0,0,0,2023,Sheffield United


Extraer partidos de la semana

In [11]:
standings_url = "https://fbref.com/en/comps/9/Premier-League-Stats"
data = requests.get(standings_url)
from bs4 import BeautifulSoup
soup = BeautifulSoup(data.text)
# Encontrar todos los elementos de los partidos
matchup = soup.find_all(class_="matchup")

for match in matchup:
    # Extraer los equipos locales y visitantes
    local_teams = match.find_all(class_="matchup-team team1")
    visit_teams = match.find_all(class_="matchup-team team2")
    raw_dates = match.find_all(class_="match-date")

local_teams_names =[]
visit_teams_names =[]
match_dates = []
for team in local_teams:
    name = team.find('a').text.strip()  # Buscar el texto dentro de la etiqueta <a>
    local_teams_names.append(name)

for team in visit_teams:
    name = team.find('a').text.strip()  
    visit_teams_names.append(name)
    
for match in raw_dates:
    date = match.text.strip()
    match_dates.append(date)
match_dates =  match_dates+ match_dates # para que coincida tamaño y orden con las otra listas extendidas
    # Cambiar a formato compatible para pandas
match_dates = [date.replace('\xa0', ' ') for date in match_dates]
match_dates = [ date+" " + year  for date in match_dates]
match_dates = pd.to_datetime(match_dates) 

for team in visit_teams_names:
    if team not in local_teams_names:
        local_teams_names.append(team)
for team in local_teams_names:
    if team not in visit_teams_names:
        visit_teams_names.append(team)


future_matches_df = pd.DataFrame({
    'team': local_teams_names,
    'opponent': visit_teams_names,
    'date': match_dates,
    'venue':["Home","Home","Away","Away"] # falta automatizar esto
})


NameError: name 'local_teams' is not defined

In [ ]:
future_matches_df

In [ ]:
df_ready_for_predictions = pd.concat([future_matches_df, match_df], ignore_index=True, sort=False)

In [ ]:
df_ready_for_predictions

In [ ]:
df_ready_for_predictions.to_csv("matchesPremier.csv")